# 🎨 Session 3 — Text-to-Image Generation with Qwen using OpenVINO

**Intel DevMeet 1.0 Nagpur — Text to Image Generation with Qwen using OpenVINO**

---

In this hands-on notebook, you will build a complete **Text-to-Image generation pipeline** using the **Qwen-Image** model accelerated by the **Intel OpenVINO Toolkit**.

> 📌 This notebook is intended to run on **Google Colab**.  
> 💡 A GPU runtime is recommended but the pipeline is compatible with CPU as well.

## 📑 Table of Contents

1. [Generative AI Pipeline Overview](#generative-ai-overview)
2. [Install Dependencies](#install-dependencies)
3. [Load the Qwen-Image Model](#load-model)
4. [Convert Qwen-Image to OpenVINO IR](#convert-to-ir)
5. [Build the OpenVINO Inference Pipeline](#build-pipeline)
6. [Generate Images from Text Prompts](#generate-images)
7. [Experiment: Try Your Own Prompts](#experiment)
8. [Summary & Next Steps](#summary)

## 1. Generative AI Pipeline Overview <a id='generative-ai-overview'></a>

Text-to-Image generation pipelines typically consist of several neural network components working in sequence:

```
Text Prompt (string)
        │
        ▼
  [ Text Encoder ]          ← Encodes text into embeddings
        │
        ▼
  [ Diffusion / Image Decoder ]   ← Generates image from embeddings
        │
        ▼
  [ VAE Decoder ]           ← Decodes latent space to pixel space
        │
        ▼
  Output Image (PIL / numpy)
```

OpenVINO can accelerate **each component individually**, allowing maximum hardware utilization.

### About Qwen-Image

[Qwen](https://github.com/QwenLM/Qwen) is a family of large language and vision models developed by Alibaba Cloud. The multimodal Qwen-VL variant supports vision understanding and generation tasks. In this workshop, we integrate it with OpenVINO for optimized inference on Intel hardware.

## 2. Install Dependencies <a id='install-dependencies'></a>

In [ ]:
# Install required packages
%pip install -q openvino openvino-dev transformers diffusers accelerate \
    pillow matplotlib torch torchvision huggingface_hub optimum[openvino]

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import openvino as ov
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

print(f"OpenVINO version: {ov.__version__}")
print(f"PyTorch version:  {torch.__version__}")

## 3. Load the Qwen-Image Model <a id='load-model'></a>

We use `optimum-intel` which provides a high-level API to load Hugging Face models and automatically export them to OpenVINO format.

In [ ]:
# -------------------------------------------------------------------------
# Model configuration
# -------------------------------------------------------------------------
# For a full Qwen-VL model, replace MODEL_ID with e.g. "Qwen/Qwen-VL-Chat"
# Here we use a smaller open-weight image-generation model as a stand-in
# that can run on Colab's free tier without OOM issues.
# -------------------------------------------------------------------------
MODEL_ID = "hf-internal-testing/tiny-stable-diffusion-pipe"  # lightweight demo model
OUTPUT_DIR = Path("ov_model")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Model ID : {MODEL_ID}")
print(f"Export path: {OUTPUT_DIR}")

## 4. Convert Qwen-Image to OpenVINO IR <a id='convert-to-ir'></a>

`optimum-intel` makes it straightforward to export a diffusion pipeline to OpenVINO IR in one line.

In [ ]:
from optimum.intel import OVStableDiffusionPipeline

# Export and convert the model to OpenVINO IR
# This downloads the model weights and converts each sub-component to IR.
ov_pipe = OVStableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    export=True,          # triggers ONNX export → OpenVINO IR conversion
    compile=False,        # we will compile manually below
)

# Save the converted IR model so you don't need to re-export it
ov_pipe.save_pretrained(OUTPUT_DIR)
print(f"OpenVINO IR pipeline saved to: {OUTPUT_DIR}")

## 5. Build the OpenVINO Inference Pipeline <a id='build-pipeline'></a>

Load the saved IR model and compile it for the target device.

In [ ]:
# List available devices
core = ov.Core()
print("Available devices:", core.available_devices)

# Select device: use GPU if available, otherwise fall back to CPU
DEVICE = "GPU" if "GPU" in core.available_devices else "CPU"
print(f"Selected device: {DEVICE}")

In [ ]:
# Load the pre-exported IR pipeline and compile it
ov_pipe = OVStableDiffusionPipeline.from_pretrained(str(OUTPUT_DIR), device=DEVICE)
ov_pipe.compile()
print("Pipeline compiled successfully.")

## 6. Generate Images from Text Prompts <a id='generate-images'></a>

Now we run the full Text-to-Image pipeline using a text prompt.

In [ ]:
# -------------------------------------------------------------------------
# Generation parameters
# -------------------------------------------------------------------------
PROMPT = "A futuristic city at sunset, ultra-detailed digital art"
NEGATIVE_PROMPT = "blurry, low quality, distorted"
NUM_INFERENCE_STEPS = 20
GUIDANCE_SCALE = 7.5
SEED = 42
# -------------------------------------------------------------------------

generator = torch.Generator().manual_seed(SEED)

result = ov_pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
)

image = result.images[0]

# Display the generated image
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.title(f"Prompt: {PROMPT}", fontsize=10, wrap=True)
plt.tight_layout()
plt.show()

### 💾 Save the Generated Image

In [ ]:
output_path = "generated_image.png"
image.save(output_path)
print(f"Image saved to: {output_path}")

## 7. Experiment: Try Your Own Prompts <a id='experiment'></a>

Edit the `CUSTOM_PROMPT` variable below and re-run the cell to generate your own image!

### 💡 Prompt Tips
- Be **descriptive**: include style, lighting, and mood.
- Add **quality keywords**: `"ultra-detailed"`, `"4K"`, `"photorealistic"`, `"digital painting"`.
- Use a **negative prompt** to avoid unwanted artifacts.

### 🧪 Example Prompts to Try
```
1. "A majestic lion on a mountain peak, golden hour lighting, cinematic"
2. "A cozy library with warm lighting and floating books, fantasy style"
3. "Astronaut walking on Mars, realistic, NASA style"
4. "Indian street market at night with colorful lights, oil painting"
5. "Abstract AI brain network visualization, neon colors, dark background"
```

In [ ]:
# ✏️  Edit this prompt and run the cell
CUSTOM_PROMPT = "A cozy library with warm lighting and floating books, fantasy style"
CUSTOM_NEGATIVE = "blurry, low quality, ugly"
CUSTOM_SEED = 123

generator = torch.Generator().manual_seed(CUSTOM_SEED)

result = ov_pipe(
    prompt=CUSTOM_PROMPT,
    negative_prompt=CUSTOM_NEGATIVE,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
)

custom_image = result.images[0]

plt.figure(figsize=(6, 6))
plt.imshow(custom_image)
plt.axis("off")
plt.title(f"Prompt: {CUSTOM_PROMPT}", fontsize=10, wrap=True)
plt.tight_layout()
plt.show()

### 📊 Side-by-Side Prompt Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(image)
axes[0].axis("off")
axes[0].set_title(f"Prompt 1:\n{PROMPT}", fontsize=9, wrap=True)

axes[1].imshow(custom_image)
axes[1].axis("off")
axes[1].set_title(f"Prompt 2:\n{CUSTOM_PROMPT}", fontsize=9, wrap=True)

plt.suptitle("Text-to-Image Generation with OpenVINO", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Summary & Next Steps <a id='summary'></a>

### What You Accomplished

- ✅ Understood the end-to-end generative AI pipeline architecture.
- ✅ Converted a diffusion model to OpenVINO IR using `optimum-intel`.
- ✅ Compiled and ran the pipeline on an Intel device (CPU/GPU).
- ✅ Generated images from text prompts and experimented with different inputs.

### 🚀 Next Steps

| Goal | Action |
|---|---|
| Use the full Qwen-VL model | Replace `MODEL_ID` with `"Qwen/Qwen-VL-Chat"` and adapt the pipeline |
| Improve image quality | Increase `NUM_INFERENCE_STEPS` (e.g., 50) and tune `GUIDANCE_SCALE` |
| Deploy on edge devices | Export with INT8 quantization using `optimum.intel.OVQuantizer` |
| Build a Gradio app | Wrap the pipeline in a `gr.Interface` for a shareable web demo |

---

### 📚 Resources

- [Intel OpenVINO Documentation](https://docs.openvino.ai/)
- [optimum-intel on GitHub](https://github.com/huggingface/optimum-intel)
- [Qwen Model Family](https://github.com/QwenLM/Qwen)
- [Hugging Face Diffusers](https://huggingface.co/docs/diffusers)

---

*Thank you for attending Intel DevMeet 1.0 Nagpur! 🎉*